In [1]:
import pandas as pd
import os
import glob
from pathlib import Path

In [2]:
ROOT = Path("../data")

RESULT_DIRS = [
    ROOT / "generation step",
]

csv_files = []
for d in RESULT_DIRS:
    csv_files.extend(sorted(d.rglob("*.csv")))

print(f"Found {len(csv_files)} CSV files")

Found 14 CSV files


In [3]:
csv_files

[PosixPath('../data/generation step/multilingual_claude_sonnet4_answers.csv'),
 PosixPath('../data/generation step/multilingual_claude_sonnet4_answers_advanced.csv'),
 PosixPath('../data/generation step/multilingual_gemini31_flash_lite_answers.csv'),
 PosixPath('../data/generation step/multilingual_gemini31_flash_lite_answers_advanced.csv'),
 PosixPath('../data/generation step/multilingual_gemma4_answers.csv'),
 PosixPath('../data/generation step/multilingual_gemma4_answers_advanced.csv'),
 PosixPath('../data/generation step/multilingual_gpt54mini_answers.csv'),
 PosixPath('../data/generation step/multilingual_gpt54mini_answers_advanced.csv'),
 PosixPath('../data/generation step/multilingual_qwen36_answers.csv'),
 PosixPath('../data/generation step/multilingual_qwen36_answers_advanced.csv'),
 PosixPath('../data/generation step/withgold/gemma4_26b__advanced__distractor_plus_gold.csv'),
 PosixPath('../data/generation step/withgold/gemma4_26b__simple__distractor_plus_gold.csv'),
 PosixPat

In [4]:
records = []

for fpath in csv_files:
    try:
        df = pd.read_csv(fpath, low_memory=False)
    except Exception as e:
        records.append({"file": str(fpath.relative_to(ROOT)), "error": str(e)})
        continue

    answer_cols = [c for c in df.columns if c.lower().startswith("answer")]
    total_rows = len(df)

    if not answer_cols:
        records.append({
            "file": str(fpath.relative_to(ROOT)),
            "total_rows": total_rows,
            "answer_columns": "NONE",
            "null_count": None,
            "null_pct": None,
            "non_null_count": None,
            "status": "NO ANSWER COLUMNS",
        })
        continue

    for col in answer_cols:
        null_count = df[col].isna().sum()
        non_null = total_rows - null_count
        null_pct = round(100 * null_count / total_rows, 1) if total_rows else 0

        if null_count == 0:
            status = "OK"
        elif null_count == total_rows:
            status = "ALL MISSING"
        else:
            status = "PARTIAL"

        records.append({
            "file": str(fpath.relative_to(ROOT)),
            "total_rows": total_rows,
            "answer_column": col,
            "non_null_count": non_null,
            "null_count": null_count,
            "null_pct": null_pct,
            "status": status,
        })

report = pd.DataFrame(records)
print(report.shape)

(14, 7)


## Full report

In [5]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
report

,file,total_rows,answer_column,non_null_count,null_count,null_pct,status
0,generation step/multilingual_claude_sonnet4_answers.csv,4200,answer_with_distractor_claude,4200,0,0.0,OK
1,generation step/multilingual_claude_sonnet4_answers_advanced.csv,4200,answer_with_distractor_claude,4200,0,0.0,OK
2,generation step/multilingual_gemini31_flash_lite_answers.csv,4200,answer_with_distractor_gemini,4200,0,0.0,OK
3,generation step/multilingual_gemini31_flash_lite_answers_advanced.csv,4200,answer_with_distractor_gemini,4200,0,0.0,OK
4,generation step/multilingual_gemma4_answers.csv,4200,answer_with_distractor,4200,0,0.0,OK
5,generation step/multilingual_gemma4_answers_advanced.csv,4200,answer_with_distractor,4200,0,0.0,OK
6,generation step/multilingual_gpt54mini_answers.csv,4200,answer_with_distractor,4200,0,0.0,OK
7,generation step/multilingual_gpt54mini_answers_advanced.csv,4200,answer_with_distractor,4200,0,0.0,OK
8,generation step/multilingual_qwen36_answers.csv,4200,answer_with_distractor,4200,0,0.0,OK
9,generation step/multilingual_qwen36_answers_advanced.csv,4200,answer_with_distractor,4200,0,0.0,OK


## Status summary

In [6]:
report["status"].value_counts()

status
OK    14
Name: count, dtype: int64

## Files with missing or incomplete generations

In [7]:
incomplete = report[report["status"].isin(["PARTIAL", "ALL MISSING", "NO ANSWER COLUMNS"])]
incomplete

,file,total_rows,answer_column,non_null_count,null_count,null_pct,status


---
## Sample consistency check
Verify that all files contain exactly the same set of samples (identified by `language` + `query_id` + `distractor_id`).

In [8]:
# Load sample keys from every file
ID_COLS = ["language", "query_id", "distractor_id"]

file_keys = {}  # file label -> frozenset of tuples
key_counts = {}  # file label -> row count

for fpath in csv_files:
    label = str(fpath.relative_to(ROOT))
    try:
        df = pd.read_csv(fpath, usecols=lambda c: c in ID_COLS, low_memory=False)
    except Exception as e:
        print(f"ERROR reading {label}: {e}")
        continue

    available = [c for c in ID_COLS if c in df.columns]
    keys = frozenset(map(tuple, df[available].values.tolist()))
    file_keys[label] = keys
    key_counts[label] = len(df)

print(f"Loaded keys from {len(file_keys)} files")

Loaded keys from 14 files


In [9]:
# Union of all sample keys across all files
all_keys = frozenset().union(*file_keys.values())
print(f"Total unique samples across all files: {len(all_keys)}")

Total unique samples across all files: 4200


In [10]:
# Per-file comparison against the full universe
consistency = []
for label, keys in file_keys.items():
    missing = all_keys - keys
    extra = keys - all_keys  # always 0 since all_keys is the union
    consistency.append({
        "file": label,
        "row_count": key_counts[label],
        "unique_samples": len(keys),
        "missing_from_universe": len(missing),
        "consistent": len(missing) == 0,
    })

consistency_df = pd.DataFrame(consistency)
consistency_df

,file,row_count,unique_samples,missing_from_universe,consistent
0,generation step/multilingual_claude_sonnet4_answers.csv,4200,4200,0,True
1,generation step/multilingual_claude_sonnet4_answers_advanced.csv,4200,4200,0,True
2,generation step/multilingual_gemini31_flash_lite_answers.csv,4200,4200,0,True
3,generation step/multilingual_gemini31_flash_lite_answers_advanced.csv,4200,4200,0,True
4,generation step/multilingual_gemma4_answers.csv,4200,4200,0,True
5,generation step/multilingual_gemma4_answers_advanced.csv,4200,4200,0,True
6,generation step/multilingual_gpt54mini_answers.csv,4200,4200,0,True
7,generation step/multilingual_gpt54mini_answers_advanced.csv,4200,4200,0,True
8,generation step/multilingual_qwen36_answers.csv,4200,4200,0,True
9,generation step/multilingual_qwen36_answers_advanced.csv,4200,4200,0,True


### Files that are NOT consistent with the full sample universe

In [11]:
inconsistent = consistency_df[~consistency_df["consistent"]]
if inconsistent.empty:
    print("All files share the same set of samples.")
else:
    display(inconsistent)

All files share the same set of samples.


### Pairwise sample overlap between files

In [12]:
labels = list(file_keys.keys())
short_labels = [Path(l).name for l in labels]

overlap_matrix = pd.DataFrame(index=short_labels, columns=short_labels, dtype=int)

for i, la in enumerate(labels):
    for j, lb in enumerate(labels):
        overlap_matrix.iloc[i, j] = len(file_keys[la] & file_keys[lb])

overlap_matrix

,multilingual_claude_sonnet4_answers.csv,multilingual_claude_sonnet4_answers_advanced.csv,multilingual_gemini31_flash_lite_answers.csv,multilingual_gemini31_flash_lite_answers_advanced.csv,multilingual_gemma4_answers.csv,multilingual_gemma4_answers_advanced.csv,multilingual_gpt54mini_answers.csv,multilingual_gpt54mini_answers_advanced.csv,multilingual_qwen36_answers.csv,multilingual_qwen36_answers_advanced.csv,gemma4_26b__advanced__distractor_plus_gold.csv,gemma4_26b__simple__distractor_plus_gold.csv,qwen3.6:27b__advanced__distractor_plus_gold.csv,qwen3.6:27b__simple__distractor_plus_gold.csv
multilingual_claude_sonnet4_answers.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_claude_sonnet4_answers_advanced.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_gemini31_flash_lite_answers.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_gemini31_flash_lite_answers_advanced.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_gemma4_answers.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_gemma4_answers_advanced.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_gpt54mini_answers.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_gpt54mini_answers_advanced.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_qwen36_answers.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0
multilingual_qwen36_answers_advanced.csv,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0,4200.0


### Which specific samples are missing per file (if any)

In [13]:
for label in inconsistent["file"].tolist():
    missing_keys = sorted(all_keys - file_keys[label])
    print(f"\n{label} — {len(missing_keys)} missing samples:")
    missing_df = pd.DataFrame(missing_keys, columns=ID_COLS[:len(missing_keys[0])])
    display(missing_df)